# 📊 NASDAQ Stock Splits Scraper

> **Objetivo**: Descargar tickers del NASDAQ con splits programados o recientes desde Yahoo Finance Calendar.
> 
> **Fuente**: https://finance.yahoo.com/calendar/splits
> 
> **Autor**: Asistente de Trading Sistemático
> 
> **Última actualización**: $(date)

## 🔧 Paso 1: Instalación de dependencias (solo primera vez)

In [1]:
# Descomenta y ejecuta esta celda si necesitas instalar las librerías
# !pip install requests pandas matplotlib openpyxl yfinance

print("✅ Si ves este mensaje, las librerías ya están instaladas o las instalaste correctamente.")

✅ Si ves este mensaje, las librerías ya están instaladas o las instalaste correctamente.


## 📦 Paso 2: Imports y configuración

In [2]:
import requests
import json
import re
import pandas as pd
from datetime import datetime, timedelta
import time
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 🎛️ Paso 3: ANÁLISIS DE SPLITS USANDO YFINANCE

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import json
import time

# ================= CONFIGURACIÓN =================
START_DATE = "2025-01-01"
END_DATE = "2026-04-30"
EXCHANGE_FILTER = "NASDAQ"  # 'NASDAQ', 'NYSE', o 'ALL'
OUTPUT_FILENAME = f"stock_splits_{EXCHANGE_FILTER.lower()}_{START_DATE}_to_{END_DATE}"

# URL de StockAnalysis (tabla de splits)
URL = 'https://stockanalysis.com/actions/splits/'

# ================= FUNCIONES =================

def get_splits_data():
    """
    Obtiene la tabla de splits desde stockanalysis.com
    """
    print(f"\n🌐 Extrayendo datos de {URL}...")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    }
    
    try:
        response = requests.get(URL, headers=headers, timeout=30)
        response.raise_for_status()
        print("✅ Página descargada correctamente")
    except Exception as e:
        print(f"❌ Error al descargar la página: {e}")
        return pd.DataFrame()
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Buscar la tabla principal
    table = soup.find('table')
    if not table:
        print("❌ No se encontró la tabla de splits")
        return pd.DataFrame()
    
    # Extraer cabeceras
    thead = table.find('thead')
    if thead:
        headers_row = thead.find_all('th')
        headers = [th.get_text(strip=True) for th in headers_row]
    else:
        # Si no hay thead, usar las primeras filas como cabecera
        first_row = table.find('tr')
        headers = [td.get_text(strip=True) for td in first_row.find_all('td')]
        # Luego eliminar esa fila de datos
    
    # Extraer filas de datos
    tbody = table.find('tbody')
    if not tbody:
        tbody = table
    
    rows_data = []
    for tr in tbody.find_all('tr'):
        # Saltar filas que no tengan celdas
        cells = tr.find_all('td')
        if not cells:
            continue
        
        row = [cell.get_text(strip=True) for cell in cells]
        rows_data.append(row)
    
    if not rows_data:
        print("❌ No se encontraron filas de datos")
        return pd.DataFrame()
    
    # Crear DataFrame
    df = pd.DataFrame(rows_data, columns=headers[:len(rows_data[0])])
    
    print(f"📊 Columnas encontradas: {list(df.columns)}")
    print(f"📈 Filas totales: {len(df)}")
    
    # Verificar columnas necesarias
    required_cols = ['Date', 'Symbol']
    for col in required_cols:
        if col not in df.columns:
            print(f"⚠️ Columna '{col}' no encontrada. Columnas disponibles: {list(df.columns)}")
            # Intentar mapear por posición si es necesario
            if 'Date' not in df.columns and len(df.columns) > 2:
                df.rename(columns={df.columns[2]: 'Date'}, inplace=True)
            if 'Symbol' not in df.columns and len(df.columns) > 0:
                df.rename(columns={df.columns[0]: 'Symbol'}, inplace=True)
    
    # Convertir fecha
    try:
        df['Date'] = pd.to_datetime(df['Date'], format='%b %d, %Y')
    except:
        try:
            df['Date'] = pd.to_datetime(df['Date'])
        except:
            print("⚠️ No se pudo convertir la fecha. Se usará como string.")
    
    return df

def filter_by_date(df, start_date, end_date):
    """Filtra el DataFrame por rango de fechas"""
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    
    mask = (df['Date'] >= start) & (df['Date'] <= end)
    filtered = df[mask].copy()
    
    print(f"📅 Splits en el rango {start_date} → {end_date}: {len(filtered)}")
    return filtered

def get_exchange_tickers(exchange):
    """
    Obtiene tickers de NASDAQ o NYSE desde fuente oficial
    """
    import requests
    
    tickers = set()
    
    if exchange == "NASDAQ":
        url = "https://www.nasdaqtrader.com/dynamic/symdir/nasdaqlisted.txt"
    elif exchange == "NYSE":
        url = "https://www.nasdaqtrader.com/dynamic/symdir/otherlisted.txt"
    else:
        return None
    
    try:
        print(f"\n📥 Descargando lista de tickers de {exchange}...")
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        for line in response.text.split('\n')[1:]:
            if '|' in line:
                symbol = line.split('|')[0].strip()
                if symbol and not symbol.startswith('$'):
                    tickers.add(symbol)
        
        print(f"✅ {len(tickers)} tickers obtenidos de {exchange}")
        return tickers
        
    except Exception as e:
        print(f"⚠️ Error descargando tickers: {e}")
        # Fallback con tickers comunes del exchange
        if exchange == "NASDAQ":
            fallback = {"AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "NFLX", "INTC", "AMD"}
            print(f"📋 Usando fallback con {len(fallback)} tickers")
            return fallback
        return set()

def filter_by_exchange(df, exchange):
    """Filtra el DataFrame por exchange"""
    if exchange == "ALL":
        return df
    
    exchange_tickers = get_exchange_tickers(exchange)
    if not exchange_tickers:
        print(f"⚠️ No se pudo filtrar por {exchange}, mostrando todos")
        return df
    
    mask = df['Symbol'].isin(exchange_tickers)
    filtered = df[mask].copy()
    
    print(f"🏢 Splits de {exchange}: {len(filtered)}/{len(df)}")
    return filtered

def save_results(df, filename):
    """Guarda el DataFrame en CSV, JSON y TXT"""
    if df.empty:
        print("\n⚠️ No hay datos para guardar")
        create_empty_report(filename)
        return
    
    # Ordenar por fecha
    df = df.sort_values('Date')
    
    # Convertir fecha a string para JSON y TXT
    df_display = df.copy()
    df_display['Date'] = df_display['Date'].dt.strftime('%Y-%m-%d')
    
    # CSV
    csv_file = f"{filename}.csv"
    df_display.to_csv(csv_file, index=False)
    print(f"\n✅ CSV guardado: {csv_file}")
    
    # JSON
    json_file = f"{filename}.json"
    records = df_display.to_dict(orient='records')
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON guardado: {json_file}")
    
    # TXT formateado
    txt_file = f"{filename}.txt"
    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("📊 REPORTE DE STOCK SPLITS\n")
        f.write("="*80 + "\n")
        f.write(f"📅 Período: {START_DATE} → {END_DATE}\n")
        f.write(f"🏢 Exchange: {EXCHANGE_FILTER}\n")
        f.write(f"📈 Total splits: {len(df)}\n")
        f.write(f"🕒 Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        
        for idx, row in df_display.iterrows():
            f.write(f"{idx+1}. {row['Symbol']}\n")
            f.write(f"   • Empresa: {row.get('Name', 'N/A')}\n")
            f.write(f"   • Fecha: {row['Date']}\n")
            if 'Ratio' in row:
                f.write(f"   • Ratio: {row['Ratio']}\n")
            if 'Type' in row:
                f.write(f"   • Tipo: {row['Type']}\n")
            f.write("\n")
    
    print(f"✅ TXT guardado: {txt_file}")
    
    # Mostrar resumen en consola
    print("\n" + "="*80)
    print("📋 SPLITS ENCONTRADOS:")
    print("="*80)
    for _, row in df_display.iterrows():
        ratio_str = f" ({row['Ratio']})" if 'Ratio' in row else ""
        type_str = f" [{row['Type']}]" if 'Type' in row else ""
        print(f"  • {row['Symbol']:<10} {row['Date']:<12} {row.get('Name', '')[:40]}{ratio_str}{type_str}")
    print("="*80)

def create_empty_report(filename):
    """Crea un reporte cuando no hay datos"""
    txt_file = f"{filename}.txt"
    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("📊 REPORTE DE STOCK SPLITS - SIN DATOS\n")
        f.write("="*80 + "\n")
        f.write(f"📅 Período: {START_DATE} → {END_DATE}\n")
        f.write(f"🏢 Exchange: {EXCHANGE_FILTER}\n")
        f.write(f"🕒 Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        f.write("⚠️ No se encontraron splits en el rango especificado.\n\n")
        f.write("💡 Posibles causas:\n")
        f.write("   • No hay splits programados en ese período\n")
        f.write("   • La fuente de datos (stockanalysis.com) puede haber cambiado\n\n")
        f.write("💡 Verifica manualmente en:\n")
        f.write("   https://stockanalysis.com/actions/splits/\n")
    
    print(f"✅ Reporte vacío creado: {txt_file}")

# ================= EJECUCIÓN PRINCIPAL =================

def main():
    print("\n" + "🚀"*40)
    print("STOCK SPLITS SCRAPER - StockAnalysis.com")
    print("🚀"*40)
    print(f"\n📅 Rango de búsqueda: {START_DATE} → {END_DATE}")
    print(f"🏢 Filtro por exchange: {EXCHANGE_FILTER}")
    
    # 1. Obtener datos de splits
    df_raw = get_splits_data()
    if df_raw.empty:
        print("\n❌ No se pudo obtener la tabla de splits")
        create_empty_report(OUTPUT_FILENAME)
        return
    
    # 2. Filtrar por fecha
    df_date_filtered = filter_by_date(df_raw, START_DATE, END_DATE)
    if df_date_filtered.empty:
        print("\n⚠️ No hay splits en el rango de fechas")
        save_results(df_date_filtered, OUTPUT_FILENAME)
        return
    
    # 3. Filtrar por exchange
    df_exchange_filtered = filter_by_exchange(df_date_filtered, EXCHANGE_FILTER)
    
    # 4. Guardar resultados
    save_results(df_exchange_filtered, OUTPUT_FILENAME)
    
    print("\n✨ ¡Proceso completado exitosamente!")

if __name__ == "__main__":
    main()


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
STOCK SPLITS SCRAPER - StockAnalysis.com
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

📅 Rango de búsqueda: 2025-01-01 → 2026-04-30
🏢 Filtro por exchange: NASDAQ

🌐 Extrayendo datos de https://stockanalysis.com/actions/splits/...
✅ Página descargada correctamente
📊 Columnas encontradas: ['Date', 'Symbol', 'Company Name', 'Type', 'Split Ratio']
📈 Filas totales: 500
📅 Splits en el rango 2025-01-01 → 2026-04-30: 500

📥 Descargando lista de tickers de NASDAQ...
✅ 5420 tickers obtenidos de NASDAQ
🏢 Splits de NASDAQ: 414/500

✅ CSV guardado: stock_splits_nasdaq_2025-01-01_to_2026-04-30.csv
✅ JSON guardado: stock_splits_nasdaq_2025-01-01_to_2026-04-30.json
✅ TXT guardado: stock_splits_nasdaq_2025-01-01_to_2026-04-30.txt

📋 SPLITS ENCONTRADOS:
  • EJH        2025-05-30    [Reverse]
  • LYEL       2025-06-02    [Reverse]
  • ARBB       2025-06-02    [Reverse]
  • BNBX       2025-06-02    [Reverse]
  • EKSO       2025-06-02    [Reverse]
  • AGMH       2025-0